# Comparing Jupytext and jupyter-plainb

[jupyter-plainb](https://github.com/notebook-link/jupyter-plainb) is a JupyterLab/JupyterLite
extension that converts plain text files to notebooks, using the parsers from the
[`plainb`](https://github.com/notebook-link/plainb) npm package. It supports a subset of the
formats that Jupytext supports: the percent format, Sphinx-Gallery scripts, classic Markdown,
and MyST Markdown.

This notebook checks how closely `plainb`'s parsers agree with Jupytext's own parsers, using
Jupytext's *mirror files* as the reference corpus: `tests/data/notebooks/outputs/` contains,
for every test notebook, the text representation that Jupytext itself produces in each format.
Since `plainb` has no Python bindings, we drive it from a small Node.js CLI
(`demo/plainb_compare/convert.mjs`) and compare its output, cell by cell, to what
`jupytext.read()` produces for the same file.

Run `npm install` in `demo/plainb_compare/` once before running this notebook (the cell below
does this automatically if needed). A `nodejs` environment is provided by this repo's pixi
environment.

In [1]:
import json
import subprocess
from pathlib import Path

import jupytext

REPO_ROOT = Path(jupytext.__file__).resolve().parents[2]
BRIDGE_DIR = REPO_ROOT / "demo" / "plainb_compare"
CONVERT_JS = BRIDGE_DIR / "convert.mjs"
MIRRORS_ROOT = REPO_ROOT / "tests" / "data" / "notebooks" / "outputs"

# mirror directory -> (jupytext format used to write it, matching plainb parser)
FORMAT_MAP = {
    "ipynb_to_percent": ("auto:percent", "parsePy"),
    "ipynb_to_sphinx": ("py:sphinx", "parseSphinxGallery"),
    "ipynb_to_myst": ("md:myst", "parseMystMd"),
    "ipynb_to_md": ("md", "parseClassicMd"),
}

if not (BRIDGE_DIR / "node_modules").exists():
    subprocess.run(["npm", "install"], cwd=BRIDGE_DIR, check=True)

## Helpers

`run_plainb` shells out to Node for a single file. `cell_source` normalizes a cell's source to
a plain string, since `plainb` stores it as a list of lines (nbformat's on-disk convention)
while `jupytext.read()` returns a single string per cell.

In [2]:
def run_plainb(parser: str, path: Path) -> tuple[dict | None, str | None]:
    proc = subprocess.run(
        ["node", str(CONVERT_JS), parser, str(path)],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        return None, proc.stderr.strip()
    return json.loads(proc.stdout), None


def cell_source(source) -> str:
    return "".join(source) if isinstance(source, list) else source


def compare_file(mirror_dir: str, filename: str, jupytext_fmt: str, plainb_parser: str) -> dict:
    path = MIRRORS_ROOT / mirror_dir / filename
    row = dict(mirror_dir=mirror_dir, file=filename)

    try:
        jt_nb = jupytext.read(path, fmt=jupytext_fmt)
    except Exception as exc:  # jupytext itself fails to read its own mirror: report, don't crash
        return {**row, "error": f"jupytext: {exc}"}

    pb_nb, error = run_plainb(plainb_parser, path)
    if error:
        return {**row, "error": f"plainb: {error}"}

    jt_cells = [(c.cell_type, c.source) for c in jt_nb.cells]
    pb_cells = [(c["cell_type"], cell_source(c["source"])) for c in pb_nb["cells"]]

    same_length = len(jt_cells) == len(pb_cells)
    type_match = same_length and all(a[0] == b[0] for a, b in zip(jt_cells, pb_cells))
    source_match = same_length and all(a[1] == b[1] for a, b in zip(jt_cells, pb_cells))

    first_diff = None
    if same_length and not source_match:
        first_diff = next(i for i, (a, b) in enumerate(zip(jt_cells, pb_cells)) if a != b)

    return {
        **row,
        "error": None,
        "jt_cells": len(jt_cells),
        "pb_cells": len(pb_cells),
        "same_length": same_length,
        "type_match": type_match,
        "source_match": source_match,
        "first_diff_cell": first_diff,
    }

## Run the comparison over every mirror file

In [3]:
results = [
    compare_file(mirror_dir, path.name, jupytext_fmt, plainb_parser)
    for mirror_dir, (jupytext_fmt, plainb_parser) in FORMAT_MAP.items()
    for path in sorted((MIRRORS_ROOT / mirror_dir).iterdir())
]
len(results)

194

## Agreement rate by format

`source_match` requires the two implementations to agree on cell count, cell type, and cell
source, in order, for every cell in the file. It is a strict, cell-exact agreement measure.

In [4]:
by_dir: dict[str, list[dict]] = {}
for row in results:
    by_dir.setdefault(row["mirror_dir"], []).append(row)

header = f"{'mirror_dir':<20} {'n_files':>8} {'errors':>7} {'same_len':>9} {'type_match':>11} {'source_match':>13} {'agreement':>10}"
print(header)
print("-" * len(header))
for mirror_dir, rows in by_dir.items():
    n = len(rows)
    errors = sum(r["error"] is not None for r in rows)
    same_length = sum(r.get("same_length", False) for r in rows)
    type_match = sum(r.get("type_match", False) for r in rows)
    source_match = sum(r.get("source_match", False) for r in rows)
    print(
        f"{mirror_dir:<20} {n:>8} {errors:>7} {same_length:>9} {type_match:>11} "
        f"{source_match:>13} {source_match / n:>10.1%}"
    )

mirror_dir            n_files  errors  same_len  type_match  source_match  agreement
------------------------------------------------------------------------------------
ipynb_to_percent           64       0        29          29            18      28.1%
ipynb_to_sphinx            11       0         0           0             0       0.0%
ipynb_to_myst              58       0        57          57            55      94.8%
ipynb_to_md                61       0        34          31            28      45.9%


## Where do they disagree?

For files where cell count/type match but sources differ, show the first mismatching cell from
each side. For files with a different cell count (or a hard error), just report as such.

In [5]:
mismatches = [r for r in results if r["error"] is not None or not r.get("source_match", False)]
print(f"{len(mismatches)} / {len(results)} files disagree")

for row in mismatches:
    print(f"\n=== {row['mirror_dir']}/{row['file']} ===")
    if row["error"]:
        print(f"  error: {row['error']}")
        continue
    if not row["same_length"]:
        print(f"  cell count differs: jupytext={row['jt_cells']} plainb={row['pb_cells']}")
        continue
    if not row["type_match"]:
        print("  cell types differ (see first_diff_cell)")
    jupytext_fmt, plainb_parser = FORMAT_MAP[row["mirror_dir"]]
    path = MIRRORS_ROOT / row["mirror_dir"] / row["file"]
    jt_nb = jupytext.read(path, fmt=jupytext_fmt)
    pb_nb, _ = run_plainb(plainb_parser, path)
    i = row["first_diff_cell"]
    jt_cell, pb_cell = jt_nb.cells[i], pb_nb["cells"][i]
    print(f"  first differing cell: #{i}")
    print(f"  jupytext [{jt_cell.cell_type}]: {jt_cell.source[:200]!r}")
    print(f"  plainb   [{pb_cell['cell_type']}]: {cell_source(pb_cell['source'])[:200]!r}")

93 / 194 files disagree

=== ipynb_to_percent/Line_breaks_in_LateX_305.py ===


  first differing cell: #2
  jupytext [markdown]: 'This cell uses the triple quote cell markers introduced at https://github.com/mwouts/jupytext/issues/305\n\n$$\n\\begin{align}\n\\dot{x} & = \\sigma(y-x)\\\\\n\\dot{y} & = \\rho x - y - xz \\\\\n\\dot{z} & = -\\beta z'
  plainb   [markdown]: "r'''\nThis cell uses the triple quote cell markers introduced at https://github.com/mwouts/jupytext/issues/305\n\n$$\n\\begin{align}\n\\dot{x} & = \\sigma(y-x)\\\\\n\\dot{y} & = \\rho x - y - xz \\\\\n\\dot{z} & = -\\b"

=== ipynb_to_percent/Notebook with html and latex cells.py ===


  first differing cell: #0
  jupytext [code]: '%%html\n<p><a href="https://github.com/mwouts/jupytext", style="color: rgb(0,0,255)">Jupytext</a> on GitHub</p>'
  plainb   [code]: '# <p><a href="https://github.com/mwouts/jupytext", style="color: rgb(0,0,255)">Jupytext</a> on GitHub</p>'

=== ipynb_to_percent/Notebook_with_R_magic.py ===


  first differing cell: #1
  jupytext [code]: '%load_ext rpy2.ipython'
  plainb   [code]: '# %load_ext rpy2.ipython'

=== ipynb_to_percent/Notebook_with_more_R_magic_111.py ===


  first differing cell: #0
  jupytext [code]: '%load_ext rpy2.ipython\nimport pandas as pd\n\ndf = pd.DataFrame(\n    {\n        "Letter": ["a", "a", "a", "b", "b", "b", "c", "c", "c"],\n        "X": [4, 3, 5, 2, 1, 7, 7, 5, 9],\n        "Y": [0, 4, 3, 6'
  plainb   [code]: '# %load_ext rpy2.ipython\nimport pandas as pd\n\ndf = pd.DataFrame(\n    {\n        "Letter": ["a", "a", "a", "b", "b", "b", "c", "c", "c"],\n        "X": [4, 3, 5, 2, 1, 7, 7, 5, 9],\n        "Y": [0, 4, 3,'

=== ipynb_to_percent/Reference Guide for Calysto Scheme.scm ===
  cell count differs: jupytext=61 plainb=1

=== ipynb_to_percent/Reference Guide for Calysto Scheme.ss ===
  cell count differs: jupytext=61 plainb=1

=== ipynb_to_percent/coconut_homepage_demo.coco ===
  cell count differs: jupytext=27 plainb=26

=== ipynb_to_percent/convert_to_py_then_test_with_update83.py ===
  cell count differs: jupytext=3 plainb=2

=== ipynb_to_percent/csharp.cs ===
  cell count differs: jupytext=7 plainb=1

=== ipynb_to_pe

...

  first differing cell: #1
  jupytext [code]: '%%R -i df\nlibrary("ggplot2")\nggplot(data = df) + geom_point(aes(x = X, y = Y, color = Letter, size = Z))'
  plainb   [markdown]: '```R magic_args="-i df"\nlibrary("ggplot2")\nggplot(data = df) + geom_point(aes(x = X, y = Y, color = Letter, size = Z))\n```'

=== ipynb_to_md/Reference Guide for Calysto Scheme.md ===
  cell types differ (see first_diff_cell)


  first differing cell: #0
  jupytext [markdown]: '<img src="images/logo-64x64.png"/>\n<h1>Reference Guide for Calysto Scheme</h1>\n\n[Calysto Scheme](https://github.com/Calysto/calysto_scheme) is a real Scheme programming language, with full support for'
  plainb   [markdown]: '<img src="images/logo-64x64.png"/>\n<h1>Reference Guide for Calysto Scheme</h1>\n\n[Calysto Scheme](https://github.com/Calysto/calysto_scheme) is a real Scheme programming language, with full support for'

=== ipynb_to_md/The flavors of raw cells.md ===
  cell count differs: jupytext=6 plainb=1

=== ipynb_to_md/coconut_homepage_demo.md ===
  cell count differs: jupytext=27 plainb=26

=== ipynb_to_md/demo_gdl_fbp.md ===
  cell count differs: jupytext=14 plainb=13

=== ipynb_to_md/frozen_cell.md ===
  cell types differ (see first_diff_cell)


  first differing cell: #1
  jupytext [code]: '# This is an frozen cell\nprint("I\'m frozen so Im not executed :(")'
  plainb   [markdown]: '```python deletable=false editable=false run_control={"frozen": true}\n# This is an frozen cell\nprint("I\'m frozen so Im not executed :(")\n```'

=== ipynb_to_md/gnuplot_notebook.md ===
  cell count differs: jupytext=5 plainb=4

=== ipynb_to_md/haskell_notebook.md ===
  cell count differs: jupytext=5 plainb=4

=== ipynb_to_md/hello_world_gonb.md ===
  cell count differs: jupytext=11 plainb=10

=== ipynb_to_md/html-demo.md ===
  cell count differs: jupytext=17 plainb=16

=== ipynb_to_md/julia_functional_geometry.md ===
  cell count differs: jupytext=6 plainb=2

=== ipynb_to_md/jupyter_with_raw_cell_in_body.md ===
  cell count differs: jupytext=3 plainb=2

=== ipynb_to_md/jupyter_with_raw_cell_on_top.md ===
  cell count differs: jupytext=3 plainb=2

=== ipynb_to_md/jupyter_with_raw_cell_with_invalid_yaml.md ===
  cell count differs: jupytext=2 plai

  first differing cell: #0
  jupytext [markdown]: '> **Note**\n> \n> `slide` layer with a `top` of `30%`'
  plainb   [markdown]: '<!-- #region @deathbeds/jupyterlab-fonts={"styles": {"": {"body[data-jp-deck-mode=\'presenting\'] &": {"right": "0", "top": "30%", "width": "25%", "z-index": 1}}}} jupyterlab-slideshow={"layer": "slide"'

=== ipynb_to_md/jupytext_replication.md ===
  cell count differs: jupytext=6 plainb=1

=== ipynb_to_md/logtalk_notebook.md ===
  cell count differs: jupytext=4 plainb=1

=== ipynb_to_md/lua_example.md ===
  cell count differs: jupytext=12 plainb=8

=== ipynb_to_md/nteract_with_parameter.md ===
  cell count differs: jupytext=4 plainb=1

=== ipynb_to_md/octave_notebook.md ===


  first differing cell: #6
  jupytext [code]: '%%python\na = 1'
  plainb   [code]: 'a = 1'

=== ipynb_to_md/plotly_graphs.md ===
  cell count differs: jupytext=5 plainb=3

=== ipynb_to_md/raw_cell_with_complex_yaml_like_content.md ===
  cell count differs: jupytext=2 plainb=1

=== ipynb_to_md/raw_cell_with_non_dict_yaml_content.md ===
  cell count differs: jupytext=2 plainb=1

=== ipynb_to_md/root_cpp.md ===
  cell count differs: jupytext=3 plainb=1

=== ipynb_to_md/sample_rise_notebook_66.md ===
  cell count differs: jupytext=3 plainb=1

=== ipynb_to_md/text_outputs_and_images.md ===
  cell count differs: jupytext=12 plainb=11

=== ipynb_to_md/wolfram.md ===
  cell count differs: jupytext=7 plainb=1

=== ipynb_to_md/xcpp_by_quantstack.md ===
  cell count differs: jupytext=60 plainb=1


## Caveats

- Only the four formats `plainb` supports are compared: percent `.py`, Sphinx-Gallery `.py`,
  classic Markdown `.md`, and MyST Markdown `.md`. Jupytext supports many more (R Markdown,
  light format, `.jl`/`.R`/other-language scripts, marimo, ...).
- `plainb`'s parsers never populate `outputs` or `execution_count` (they parse text only, no
  execution), so this notebook does not compare outputs — only the cell structure and source.
- Notebook/cell *metadata* (beyond `cell_type`/`source`) is not compared either: the two
  projects use different metadata schemas (e.g. cell ids, kernelspec placement), so a
  metadata-level diff would mostly measure schema differences rather than parsing agreement.